In [2]:
"""
Simulador de Neural Architecture Search (NAS)
=============================================
Implementa as três dimensões do survey de Elsken, Metzen & Hutter (2019):

  1. Espaço de busca        -> SearchSpace (espaço baseado em células, estilo Zoph et al.)
  2. Estratégia de busca    -> RandomSearch, HillClimbing, AgingEvolution, ReinforceNAS
  3. Estimação de desempenho-> PerformanceEstimator (benchmark sintético com
                               fidelidades múltiplas e contabilização de custo)

O "desempenho verdadeiro" de cada arquitetura vem de uma função substituta
determinística (como em NAS-Bench-101), portanto nenhuma rede é treinada de fato
e o experimento roda em segundos. Dependências: numpy e matplotlib.

Uso:  python nas_simulacao.py
"""
from __future__ import annotations

import os
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OPS = ["conv3x3", "conv5x5", "sep_conv3x3", "max_pool3x3", "identity", "skip"]
N_BLOCKS = 5          # blocos por célula
# Custo relativo (proxy de FLOPs) de cada operação, útil para NAS multiobjetivo
OP_COST = np.array([1.0, 2.2, 0.5, 0.1, 0.0, 0.0])


# ----------------------------------------------------------------------------
# 1. ESPAÇO DE BUSCA
# ----------------------------------------------------------------------------
class SearchSpace:
    """Célula com N_BLOCKS blocos. Cada bloco escolhe 2 entradas e 1 operação por
    entrada. A entrada pode ser a saída de qualquer bloco anterior ou das duas
    entradas da célula (índices 0 e 1) -> espaço CONDICIONAL: o número de opções de
    entrada do bloco b é b + 2. Gene = [in1, op1, in2, op2] por bloco (20 genes)."""

    def __init__(self):
        self.cardinalities = []
        for b in range(N_BLOCKS):
            self.cardinalities += [b + 2, len(OPS), b + 2, len(OPS)]
        self.dim = len(self.cardinalities)

    def size(self) -> int:
        return int(np.prod(self.cardinalities, dtype=object))

    def sample(self, rng) -> tuple:
        return tuple(int(rng.integers(c)) for c in self.cardinalities)

    def mutate(self, arch: tuple, rng) -> tuple:
        """Mutação local: altera um único gene (análogo a trocar uma operação/conexão)."""
        a = list(arch)
        i = int(rng.integers(self.dim))
        choices = [v for v in range(self.cardinalities[i]) if v != a[i]]
        a[i] = int(rng.choice(choices))
        return tuple(a)

    def neighbors(self, arch: tuple):
        for i, c in enumerate(self.cardinalities):
            for v in range(c):
                if v != arch[i]:
                    n = list(arch)
                    n[i] = v
                    yield tuple(n)

    def cost(self, arch: tuple) -> float:
        """Custo computacional relativo da arquitetura (soma dos custos das operações)."""
        return float(sum(OP_COST[arch[i]] for i in range(1, self.dim, 2)))


# ----------------------------------------------------------------------------
# 3. ESTIMAÇÃO DE DESEMPENHO (benchmark sintético)
# ----------------------------------------------------------------------------
class PerformanceEstimator:
    """Simula treino+validação. `fidelity` = nº de épocas (até max_epochs).

    - Acurácia verdadeira: função fixa e determinística da arquitetura, com efeitos
      por gene e interações entre genes vizinhos (rugosidade que dificulta a busca).
    - Baixa fidelidade: subestima a acurácia (viés) e adiciona ruído que embaralha o
      ranking, mas ambos diminuem com mais épocas (cf. Seção 4 do artigo).
    - Custo: contabilizado em "épocas-GPU" proporcionais ao custo da arquitetura.
    """

    def __init__(self, space: SearchSpace, max_epochs: int = 40, seed: int = 0):
        self.space, self.max_epochs = space, max_epochs
        g = np.random.default_rng(seed)  # semente fixa => o "benchmark" é o mesmo sempre
        mx = max(space.cardinalities)
        self.main = g.normal(0, 1, (space.dim, mx))
        self.inter = g.normal(0, 0.8, (space.dim, mx, mx))
        self.noise_rng = np.random.default_rng(seed + 1)
        self.gpu_epochs = 0.0     # custo acumulado
        self.n_evals = 0
        self._cache = {}

    def true_accuracy(self, arch: tuple) -> float:
        if arch in self._cache:
            return self._cache[arch]
        s = sum(self.main[i, v] for i, v in enumerate(arch))
        s += sum(self.inter[i, arch[i], arch[i + 1]] for i in range(len(arch) - 1))
        s = s / np.sqrt(len(arch))
        # penaliza células quase sem operações com parâmetros (só pooling/skip)
        n_param_ops = sum(arch[i] in (0, 1, 2) for i in range(1, len(arch), 2))
        s -= 1.5 * max(0, 3 - n_param_ops)
        acc = 0.80 + 0.165 / (1 + np.exp(-s))        # faixa ~ [0.80, 0.965]
        self._cache[arch] = float(acc)
        return self._cache[arch]

    def evaluate(self, arch: tuple, fidelity: int | None = None) -> float:
        """Retorna a acurácia de validação estimada e acumula o custo."""
        f = self.max_epochs if fidelity is None else min(fidelity, self.max_epochs)
        frac = f / self.max_epochs
        true = self.true_accuracy(arch)
        bias = 0.12 * (1 - frac) ** 1.5                      # subestimação
        noise = self.noise_rng.normal(0, 0.004 + 0.03 * (1 - frac))
        self.gpu_epochs += f * (0.5 + 0.1 * self.space.cost(arch))
        self.n_evals += 1
        return float(true - bias + noise)


# ----------------------------------------------------------------------------
# 2. ESTRATÉGIAS DE BUSCA
# ----------------------------------------------------------------------------
class Searcher:
    """Interface comum. `run` devolve a trajetória (custo_acumulado, melhor_verdadeira)."""

    name = "base"

    def __init__(self, space, estimator, seed=0, fidelity=None):
        self.space, self.est, self.fid = space, estimator, fidelity
        self.rng = np.random.default_rng(seed)
        self.history = []   # (arch, acc_estimada)

    def _eval(self, arch):
        acc = self.est.evaluate(arch, self.fid)
        self.history.append((arch, acc))
        return acc

    def trajectory(self):
        """Melhor acurácia VERDADEIRA entre as arquiteturas que a busca considera
        melhores segundo a estimativa (é o que o usuário implantaria)."""
        best_est, best_arch, out = -1, None, []
        cost = 0.0
        return best_est, best_arch, out, cost

    def run(self, budget_gpu_epochs: float):
        raise NotImplementedError


class RandomSearch(Searcher):
    name = "Random Search"

    def run(self, budget):
        while self.est.gpu_epochs < budget:
            self._eval(self.space.sample(self.rng))


class HillClimbing(Searcher):
    """Sobe pelo vizinho (1 mutação) melhor entre uma amostra de vizinhos; reinicia
    aleatoriamente ao atingir um ótimo local (cf. Elsken et al., 2017)."""
    name = "Hill Climbing"

    def run(self, budget, n_neighbors=8):
        cur = self.space.sample(self.rng)
        cur_acc = self._eval(cur)
        while self.est.gpu_epochs < budget:
            improved = False
            for _ in range(n_neighbors):
                if self.est.gpu_epochs >= budget:
                    return
                cand = self.space.mutate(cur, self.rng)
                acc = self._eval(cand)
                if acc > cur_acc:
                    cur, cur_acc, improved = cand, acc, True
                    break
            if not improved:   # ótimo local aparente -> reinício
                cur = self.space.sample(self.rng)
                cur_acc = self._eval(cur)


class AgingEvolution(Searcher):
    """Evolução com torneio e remoção do indivíduo MAIS VELHO (Real et al., 2019)."""
    name = "Aging Evolution"

    def run(self, budget, pop_size=30, tournament=8):
        pop = []
        while len(pop) < pop_size and self.est.gpu_epochs < budget:
            a = self.space.sample(self.rng)
            pop.append((a, self._eval(a)))
        while self.est.gpu_epochs < budget:
            idx = self.rng.choice(len(pop), size=min(tournament, len(pop)), replace=False)
            parent = max((pop[i] for i in idx), key=lambda t: t[1])
            child = self.space.mutate(parent[0], self.rng)
            pop.append((child, self._eval(child)))
            pop.pop(0)  # remove o mais velho (não o pior)


class ReinforceNAS(Searcher):
    """RL como bandit sem estado (visão do artigo, Seção 3): a política é um produto
    de distribuições categóricas independentes (uma por gene), treinada com REINFORCE
    e baseline de média móvel. Versão simplificada do controlador RNN de Zoph & Le."""
    name = "REINFORCE (RL)"

    def run(self, budget, lr=0.08, batch=8, baseline_decay=0.9):
        logits = [np.zeros(c) for c in self.space.cardinalities]
        baseline = None
        while self.est.gpu_epochs < budget:
            samples, rewards = [], []
            for _ in range(batch):
                if self.est.gpu_epochs >= budget:
                    break
                arch = tuple(int(self.rng.choice(len(l), p=_softmax(l))) for l in logits)
                samples.append(arch)
                rewards.append(self._eval(arch))
            if not samples:
                break
            m = float(np.mean(rewards))
            baseline = m if baseline is None else baseline_decay * baseline + (1 - baseline_decay) * m
            for arch, r in zip(samples, rewards):
                adv = (r - baseline) / 0.02       # escala da vantagem
                for i, v in enumerate(arch):
                    p = _softmax(logits[i])
                    grad = -p
                    grad[v] += 1.0                # grad de log pi(v) em relação aos logits
                    logits[i] = logits[i] + lr * adv * grad / len(samples)


def _softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()


# ----------------------------------------------------------------------------
# EXPERIMENTO
# ----------------------------------------------------------------------------
def best_true_curve(searcher: Searcher, est: PerformanceEstimator, grid, costs):
    """Para cada ponto do orçamento, a acurácia VERDADEIRA da arquitetura com melhor
    estimativa vista até então (o que de fato seria entregue ao usuário)."""
    best_est, best_true, ci = -1.0, 0.0, 0
    curve = []
    for (arch, acc), c in zip(searcher.history, costs):
        if acc > best_est:
            best_est, best_true = acc, est.true_accuracy(arch)
        curve.append((c, best_true))
    out, j = [], 0
    for g in grid:
        while j + 1 < len(curve) and curve[j + 1][0] <= g:
            j += 1
        out.append(curve[j][1] if curve and curve[0][0] <= g else np.nan)
    return np.array(out)


def run_experiment(strategies, budget=6000, fidelity=None, n_seeds=10, label=""):
    space = SearchSpace()
    grid = np.linspace(budget * 0.02, budget, 60)
    results = {}
    for S in strategies:
        curves = []
        for seed in range(n_seeds):
            est = PerformanceEstimator(space, seed=0)       # mesmo benchmark
            est.noise_rng = np.random.default_rng(100 + seed)
            s = S(space, est, seed=seed, fidelity=fidelity)
            # registra o custo acumulado após cada avaliação
            costs, orig = [], s._eval

            def wrapped(arch, s=s, est=est, costs=costs, orig=orig):
                r = orig(arch)
                costs.append(est.gpu_epochs)
                return r

            s._eval = wrapped
            s.run(budget)
            curves.append(best_true_curve(s, est, grid, costs))
        results[S.name] = np.array(curves)
    return grid, results


def plot(grid, results, title, path):
    plt.figure(figsize=(8, 5))
    for name, c in results.items():
        mean, std = np.nanmean(c, 0), np.nanstd(c, 0)
        plt.plot(grid, mean, label=name, lw=2)
        plt.fill_between(grid, mean - std, mean + std, alpha=0.15)
    plt.xlabel("Custo acumulado (épocas-GPU simuladas)")
    plt.ylabel("Acurácia verdadeira da melhor arquitetura encontrada")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    plt.close()


if __name__ == "__main__":
    # Garante que o diretório para salvar os outputs exista
    os.makedirs("/mnt/user-data/outputs", exist_ok=True)

    space = SearchSpace()
    print(f"Genes por célula: {space.dim} | Tamanho do espaço: {space.size():.3e} arquiteturas")

    # Referência: melhor acurácia numa amostra grande (aproxima o ótimo do benchmark)
    ref = PerformanceEstimator(space)
    r = np.random.default_rng(999)
    best_ref = max(ref.true_accuracy(space.sample(r)) for _ in range(100000))
    print(f"Melhor acurácia verdadeira em 100k amostras aleatórias: {best_ref:.4f}\n")

    strategies = [RandomSearch, HillClimbing, AgingEvolution, ReinforceNAS]

    # Experimento A: todas avaliadas com treino completo (40 épocas)
    grid, res = run_experiment(strategies, budget=8000, fidelity=None, n_seeds=10)
    plot(grid, res, "NAS simulado - treino completo (40 épocas/arquitetura)", "/mnt/user-data/outputs/nas_alta_fidelidade.png")
    print("A) Alta fidelidade (40 épocas) - acurácia verdadeira final (média ± desvio, 10 sementes)")
    for n, c in res.items():
        print(f"   {n:<16} {np.nanmean(c[:, -1]):.4f} ± {np.nanstd(c[:, -1]):.4f}")

    # Experimento B: baixa fidelidade (5 épocas) -> mais avaliações, porém estimativa enviesada/ruidosa
    grid, res = run_experiment(strategies, budget=8000, fidelity=5, n_seeds=10)
    plot(grid, res, "NAS simulado - baixa fidelidade (5 épocas/arquitetura)", "/mnt/user-data/outputs/nas_baixa_fidelidade.png")
    print("\nB) Baixa fidelidade (5 épocas) - acurácia verdadeira final")
    for n, c in res.items():
        print(f"   {n:<16} {np.nanmean(c[:, -1]):.4f} ± {np.nanstd(c[:, -1]):.4f}")


Genes por célula: 20 | Tamanho do espaço: 3.135e+13 arquiteturas
Melhor acurácia verdadeira em 100k amostras aleatórias: 0.9640

A) Alta fidelidade (40 épocas) - acurácia verdadeira final (média ± desvio, 10 sementes)
   Random Search    0.9550 ± 0.0029
   Hill Climbing    0.9592 ± 0.0015
   Aging Evolution  0.9616 ± 0.0016
   REINFORCE (RL)   0.9567 ± 0.0041

B) Baixa fidelidade (5 épocas) - acurácia verdadeira final
   Random Search    0.9500 ± 0.0081
   Hill Climbing    0.9456 ± 0.0108
   Aging Evolution  0.9542 ± 0.0053
   REINFORCE (RL)   0.9552 ± 0.0097
